In [ ]:


import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from tqdm import tqdm


CSV_PATH = "juliet_cwe_dataset.csv"
MODEL_NAME = "Salesforce/codet5-small"
MAX_LEN = 256
BATCH_SIZE = 4
EPOCHS = 2
LR = 5e-5

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


df = pd.read_csv(CSV_PATH)

df = df[["code_clean", "cwe_index"]].dropna()
df["cwe_index"] = df["cwe_index"].astype(int)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["cwe_index"],
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))

# CLASS WEIGHTS 
class_counts = train_df["cwe_index"].value_counts().sort_index()
weights = 1.0 / class_counts
weights = weights / weights.sum()

weights_tensor = torch.tensor(weights.values, dtype=torch.float).to(device)

criterion = CrossEntropyLoss(weight=weights_tensor)

print("Class weights ready.")


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class JulietDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        )

        item = {k: torch.tensor(v) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))

        return item


train_dataset = JulietDataset(
    train_df["code_clean"],
    train_df["cwe_index"],
    tokenizer
)

val_dataset = JulietDataset(
    val_df["code_clean"],
    val_df["cwe_index"],
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


num_labels = df["cwe_index"].nunique()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

model.to(device)

optimizer = AdamW(model.parameters(), lr=LR)

print("Model ready.")

def train_epoch():
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc="Training"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate():
    model.eval()

    preds_all, labels_all = [], []
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(labels_all, preds_all, digits=4))

    return total_loss / len(val_loader)



for epoch in range(EPOCHS):

    print(f"\n===== EPOCH {epoch+1}/{EPOCHS} =====")

    train_loss = train_epoch()
    print("Train Loss:", train_loss)

    val_loss = evaluate()
    print("Val Loss:", val_loss)



SAVE_DIR = "codet5_weighted_cwe"

os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model saved to:", SAVE_DIR)